<a href="https://colab.research.google.com/github/Sood-Dhruv/localizegraph-ai/blob/main/LLM_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Main Language for translation: English  
Languages to be used: Hindi, German

In [1]:
#Cell 1.1
LANGUAGES = {
    "english": {"iso": "eng", "mms_code": "eng"},
    "hindi":   {"iso": "hin", "mms_code": "hin"},
    "german":  {"iso": "deu", "mms_code": "deu"},
}

SOURCE_LANG = "english"
TARGET_LANGS = ["hindi", "german"]

Installing dependencies:

In [ ]:
#Cell 1.2
!pip install --upgrade -qqq uv
!uv pip install -qqq unsloth vllm
!uv pip install -qqq transformers accelerate xformers bitsandbytes
!uv pip install -qqq networkx gradio openai soundfile scipy
!pip install -q sentencepiece torch torchaudio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 66.9 MB/s eta 0:00:00


Verify GPU:

In [ ]:
#Cell 1.3
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# Step 3

Expand Cultural Knwoledge Graph

In [ ]:
#Cell 2
import networkx as nx

CULTURAL_KNOWLEDGE = [
    {
        "concept": "cup_of_coffee", "type": "idiom",
        "en_text": "cheap daily expense",
        "adaptations": {
            "hin": {"text": "चाय का खर्चा",
                    "context": "India: tea (chai), not coffee, is the everyday cheap-expense comparison"},
            "deu": {"text": "Brötchengeld",
                    "context": "Germany: colloquial term for 'pocket change' / trivial cost"},
        },
    },
    {
        "concept": "kick_the_bucket", "type": "idiom",
        "en_text": "to die (euphemism)",
        "adaptations": {
            "hin": {"text": "स्वर्ग सिधारना",
                    "context": "Respectful Hindi euphemism: 'departed for heaven'"},
            "deu": {"text": "den Löffel abgeben",
                    "context": "German idiom, literally 'to hand over the spoon'"},
        },
    },
    {
        "concept": "raining_cats_and_dogs", "type": "idiom",
        "en_text": "heavy rain",
        "adaptations": {
            "hin": {"text": "मुसलाधार बारिश",
                    "context": "Standard Hindi term for torrential/heavy rain"},
            "deu": {"text": "es regnet wie aus Eimern",
                    "context": "German idiom, literally 'raining as if from buckets'"},
        },
    },
    {
        "concept": "piece_of_cake", "type": "idiom",
        "en_text": "something very easy",
        "adaptations": {
            "hin": {"text": "बाएं हाथ का खेल",
                    "context": "Literally 'a game for the left hand' — common way to say effortless"},
            "deu": {"text": "ein Kinderspiel",
                    "context": "Literally 'child's play' — standard German equivalent"},
        },
    },
    {
        "concept": "break_a_leg", "type": "idiom",
        "en_text": "good luck (esp. before a performance)",
        "adaptations": {
            "hin": {"text": "ऑल द बेस्ट",
                    "context": "No native superstition-idiom equivalent exists; speakers default to a direct 'all the best', often in English"},
            "deu": {"text": "Hals- und Beinbruch",
                    "context": "Near-literal equivalent: 'break your neck and leg' — rare case where idiom structure survives across languages"},
        },
    },
    {
        "concept": "not_the_best_option", "type": "idiom",
        "en_text": "not great, not the ideal choice",
        "adaptations": {
            "hin": {"text": "दाल में कुछ काला है",
                    "context": "APPROXIMATE MATCH ONLY — closer to 'something's off' than 'not ideal'; flag for human review in real use"},
            "deu": {"text": "nicht das Gelbe vom Ei",
                    "context": "Literally 'not the yellow of the egg' — standard German way to say not the best option"},
        },
    },
    {
        "concept": "ten_dollars", "type": "currency",
        "en_text": "monetary reference",
        "adaptations": {
            "hin": {"text": "₹800",
                    "context": "Convert to INR at market rate — preserve the 'small amount' framing over FX precision"},
            "deu": {"text": "€9",
                    "context": "Convert to EUR — same framing-over-precision rule applies"},
        },
    },
    {
        "concept": "ten_miles", "type": "unit",
        "en_text": "distance reference",
        "adaptations": {
            "hin": {"text": "16 किलोमीटर", "context": "India uses metric (km) — straightforward unit conversion"},
            "deu": {"text": "16 Kilometer", "context": "Germany uses metric (km) — same conversion logic"},
        },
    },
]


def build_cultural_kg(knowledge_base=CULTURAL_KNOWLEDGE):
    """Builds an in-memory directed Cultural Knowledge Graph from structured data."""
    G = nx.DiGraph()
    for item in knowledge_base:
        source = item["concept"]
        G.add_node(source, type=item["type"], concept=item["en_text"])
        for lang, data in item["adaptations"].items():
            target_node = f"{source}__{lang}"
            G.add_node(target_node, lang=lang, text=data["text"], context=data["context"])
            G.add_edge(source, target_node, relation="localized_adaptation", lang=lang)
    return G


def get_localization_context(graph, source_term, target_lang):
    """Returns a prompt-ready context string for a concept + target language."""
    if not graph.has_node(source_term):
        return ""
    for neighbor in graph.neighbors(source_term):
        edge_data = graph.get_edge_data(source_term, neighbor)
        if edge_data and edge_data.get("lang") == target_lang:
            node_data = graph.nodes[neighbor]
            return (f"[Cultural Context Adapter: Replace '{source_term}' with native concept "
                    f"'{node_data['text']}' to align with: {node_data['context']}]")
    return ""


def get_all_contexts_for_text(graph, detected_concepts, target_lang):
    """Given concept names detected in input text, return all matching context strings."""
    return [c for c in (get_localization_context(graph, concept, target_lang)
                        for concept in detected_concepts) if c]


# --- Quick verification ---
kg = build_cultural_kg()
print(get_localization_context(kg, "cup_of_coffee", "hin"))
print(get_localization_context(kg, "break_a_leg", "deu"))

# Step 4 — the prompting pipeline with LLM-based concept detection.

Hugging Face login:

In [ ]:
#Cell 3
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

Loading model locally:

In [ ]:
#Cell 4
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, json, re

# Login — needed for the gated meta-llama repo; not required for the unsloth pre-quantized repo below
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
# Faster alternative (pre-quantized, ~5.7GB download instead of ~16GB):
# MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")

def call_llm(prompt, max_new_tokens=300):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    attention_mask = torch.ones_like(inputs)
    output = model.generate(
        inputs,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response.strip()

Concept catalog (built from CULTURAL_KNOWLEDGE) + LLM-based detection

In [ ]:
#Cell 5
CONCEPT_CATALOG = [
    {"id": item["concept"], "description": item["en_text"]}
    for item in CULTURAL_KNOWLEDGE
]

def detect_concepts(text, catalog=CONCEPT_CATALOG):
    """Asks the LLM which catalog concepts are present in the input text. Returns a list of concept ids."""
    catalog_str = "\n".join(f'- {c["id"]}: {c["description"]}' for c in catalog)
    prompt = f"""You are an entity detector. Below is a catalog of cultural concepts/idioms.

CATALOG:
{catalog_str}

TEXT: "{text}"

Identify which catalog concepts (by id) are present or implied in the text above.
Respond with ONLY a JSON array of matching ids, nothing else. Example: ["cup_of_coffee", "ten_miles"]
If none match, respond with: []"""

    raw = call_llm(prompt, max_new_tokens=100)
    match = re.search(r"\[.*?\]", raw, re.DOTALL)
    if not match:
        return []
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return []

Full adaptation pipeline — detect concepts, inject KG context, generate adapted translation

In [ ]:
#Cell 6
FEW_SHOT_EXAMPLE = """Example:
Input (English): "It only costs a cup of coffee a day."
Context hints: [Cultural Context Adapter: Replace 'cup_of_coffee' with native concept 'चाय का खर्चा' to align with: India: tea (chai), not coffee, is the everyday cheap-expense comparison]
Output (Hindi): "इसकी कीमत रोज़ाना सिर्फ़ एक कप चाय जितनी है।"
"""

def generate_adapted_translation(text, target_lang, kg=None):
    if kg is None:
        kg = build_cultural_kg()

    detected = detect_concepts(text)
    context_hints = get_all_contexts_for_text(kg, detected, target_lang)
    hints_block = "\n".join(context_hints) if context_hints else "(no special cultural adaptations detected)"

    lang_name = "Hindi" if target_lang == "hin" else "German"

    prompt = f"""You are a culturally-aware translator. Translate the text into {lang_name}, naturally adapting
any idioms, currency, or units using the cultural context hints provided — don't translate them literally.

{FEW_SHOT_EXAMPLE}

Now translate this. Respond with ONLY the translated sentence in {lang_name} — no labels, no quotation marks, no explanation, nothing else.

Input (English): "{text}"
Context hints:
{hints_block}
Translation:"""

    raw = call_llm(prompt, max_new_tokens=100)
    cleaned = raw.split("\n")[0].strip().strip('"')
    return cleaned


# --- Quick test ---
kg = build_cultural_kg()
result = generate_adapted_translation("It only costs a cup of coffee a day.", "hin", kg=kg)
print(result)

result2 = generate_adapted_translation("Don't worry, break a leg tonight!", "deu", kg=kg)
print(result2)

### Step 5 - MMS-TTS

MMS-TTS Pipeline (Hindi + German audio generation)

In [ ]:
#Cell 7
!pip install -q -U uroman

import torch
import scipy.io.wavfile
from transformers import VitsModel, AutoTokenizer
from IPython.display import Audio, display

TTS_MODEL_IDS = {
    "hin": "facebook/mms-tts-hin",
    "deu": "facebook/mms-tts-deu",
}

_tts_cache = {}  # caches loaded (model, tokenizer) per language so we don't reload on every call

def load_tts(lang_code):
    if lang_code not in _tts_cache:
        model_id = TTS_MODEL_IDS[lang_code]
        tok = AutoTokenizer.from_pretrained(model_id)
        mdl = VitsModel.from_pretrained(model_id).to("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Loaded TTS model for '{lang_code}' — uroman required: {getattr(tok, 'is_uroman', False)}")
        _tts_cache[lang_code] = (mdl, tok)
    return _tts_cache[lang_code]


def text_to_speech(text, target_lang, output_path=None):
    """
    Converts text to speech for the given target language ('hin' or 'deu').
    Saves a .wav file and displays an inline audio player in Colab.
    Romanization for Hindi is handled automatically by the tokenizer (uroman installed above).
    """
    model, tokenizer = load_tts(target_lang)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model(**inputs).waveform

    waveform = output.float().cpu().numpy()[0]

    if output_path is None:
        output_path = f"/content/output_{target_lang}.wav"

    scipy.io.wavfile.write(output_path, rate=model.config.sampling_rate, data=waveform)
    display(Audio(waveform, rate=model.config.sampling_rate))
    return output_path


# --- Quick test using outputs from the Cell 6 pipeline ---
hin_text = generate_adapted_translation("It only costs a cup of coffee a day.", "hin", kg=kg)
deu_text = generate_adapted_translation("Don't worry, break a leg tonight!", "deu", kg=kg)

print("Hindi text:", hin_text)
hin_path = text_to_speech(hin_text, "hin")
print("Saved to:", hin_path)

print("German text:", deu_text)
deu_path = text_to_speech(deu_text, "deu")
print("Saved to:", deu_path)

# Step 6 - Evaluation by LLM

LLM-as-a-Judge Evaluation Framework

In [ ]:
#Cell 8
import pandas as pd

EVAL_DATASET = [
    "It only costs a cup of coffee a day.",
    "Don't worry, break a leg tonight!",
    "It's raining cats and dogs outside.",
    "That coding task was a piece of cake.",
    "The subscription is about ten dollars.",
    "The server room is ten miles away from here.",
    "Sadly, the old machine kicked the bucket.",
    "Honestly, that framework is not the best option."
]

def evaluate_translation(source_text, translated_text, lang_id):
    prompt = f"""You are an expert bilingual linguist evaluating a localized translation.
Source (English): "{source_text}"
Translation ({lang_id}): "{translated_text}"

Rate the translation on a scale of 1 to 5 for "Cultural Naturalness" (how idiomatic it sounds to a native speaker, rather than just a literal word-for-word translation).
Respond ONLY with a single digit from 1 to 5. No other text."""

    score = call_llm(prompt, max_new_tokens=10)
    # Extract just the first digit found in the response
    digit_match = re.search(r'\d', score)
    return int(digit_match.group()) if digit_match else None

def run_evaluation_suite(target_lang):
    print(f"--- Running Eval Suite for {target_lang.upper()} ---")
    results = []
    for text in EVAL_DATASET:
        # 1. Translate
        translation = generate_adapted_translation(text, target_lang, kg=kg)
        # 2. Evaluate
        score = evaluate_translation(text, translation, target_lang)
        results.append({
            "Source": text,
            "Translation": translation,
            "Naturalness_Score": score
        })

    df = pd.DataFrame(results)
    print(f"Average {target_lang.upper()} Naturalness Score: {df['Naturalness_Score'].mean()}/5.0")
    display(df)
    return df

# Run the benchmark
hin_eval_df = run_evaluation_suite("hin")
deu_eval_df = run_evaluation_suite("deu")

# Step 7 - UI

Gradio UICopilot Dashboard

In [ ]:
#Cell 9
import gradio as gr

def copilot_generate(text, lang_choice):
    target_lang = "hin" if lang_choice == "Hindi" else "deu"

    # 1. Get Graph Context for Telemetry
    detected = detect_concepts(text)
    context_hints = get_all_contexts_for_text(kg, detected, target_lang)
    telemetry = "\n".join(context_hints) if context_hints else "No special idioms detected. Standard translation applied."

    # 2. Generate Translation
    translation = generate_adapted_translation(text, target_lang, kg=kg)

    # 3. Synthesize Audio
    audio_path = text_to_speech(translation, target_lang)

    return translation, audio_path, telemetry

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌍 LocalizeGraph AI Copilot")
    gr.Markdown("Context-aware translation & TTS powered by an in-memory Cultural Knowledge Graph, Llama 3.1, and Meta MMS.")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(label="English Source Content", lines=3, placeholder="Enter marketing copy or idioms here...")
            lang_dropdown = gr.Dropdown(choices=["Hindi", "German"], value="Hindi", label="Target Language")
            generate_btn = gr.Button("Generate Localized Content", variant="primary")

            gr.Markdown("### Knowledge Graph Telemetry")
            telemetry_out = gr.Textbox(label="Graph Injections", lines=2, interactive=False)

        with gr.Column():
            gr.Markdown("### Copilot Output")
            translation_out = gr.Textbox(label="Culturally Adapted Text", lines=3, interactive=False)
            audio_out = gr.Audio(label="Synthesized Voiceover (MMS-TTS)", type="filepath")

    generate_btn.click(
        fn=copilot_generate,
        inputs=[input_text, lang_dropdown],
        outputs=[translation_out, audio_out, telemetry_out]
    )

demo.launch(share=True)